### Imports

In [1]:
import os, shutil
import time
import sqlite3
import csv
from psutil import virtual_memory
import pandas as pd
import requests

In [2]:
def mem():
    print(f'used memory : {round(virtual_memory()[3]/(1024*1024*1024)*10)/10}Go')

In [3]:
def stats(): 
    print("--- %s seconds ---" % (time.time() - start_time))
    mem()

In [4]:
def global_stats(): 
    print("--- %s seconds ---" % (time.time() - global_start_time))
    mem()

In [5]:
mem()

used memory : 19.5Go


In [6]:
global_start_time = time.time()

### Path set-up

In [7]:
if "DATA_DIR" not in locals():
    DATA_DIR = "./data/"
else:
    print(DATA_DIR)

if os.path.exists(DATA_DIR) and os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)

In [8]:
if "OUTPUT_DATA_FOLDER" not in locals():
    OUTPUT_DATA_FOLDER = "./output/"
else:
    print(OUTPUT_DATA_FOLDER)

if os.path.exists(OUTPUT_DATA_FOLDER) and os.path.isdir(OUTPUT_DATA_FOLDER):
    shutil.rmtree(OUTPUT_DATA_FOLDER)
os.makedirs(os.path.dirname(OUTPUT_DATA_FOLDER), exist_ok=True)

## SQLITE


In [9]:
!rm sirene.db

In [10]:
connection = sqlite3.connect('sirene.db')

In [11]:
cursor = connection.cursor()

## Unité Légale

In [12]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS unite_legale
    (
        siren,
        date_creation_unite_legale,
        sigle,
        prenom,
        identifiant_association_unite_legale,
        tranche_effectif_salarie_unite_legale,
        date_mise_a_jour_unite_legale,
        categorie_entreprise,
        etat_administratif_unite_legale,
        nom,
        nom_usage,
        nom_raison_sociale,
        nature_juridique_unite_legale,
        activite_principale_unite_legale,
        economie_sociale_solidaire_unite_legale
    )
''')

In [13]:
cursor.execute('''
                CREATE UNIQUE INDEX index_siren
                ON unite_legale (siren);
                ''')

In [14]:
connection.commit()

### Download StockUniteLegale_utf8

In [15]:
start_time = time.time()
url = 'https://files.data.gouv.fr/insee-sirene/StockUniteLegale_utf8.zip'
r = requests.get(url, allow_redirects=True)
open('StockUniteLegale_utf8.zip', 'wb').write(r.content)
shutil.unpack_archive('StockUniteLegale_utf8.zip', './')
stats()

--- 35.1636745929718 seconds ---
used memory : 20.3Go


In [16]:
df_iterator = pd.read_csv(
    'StockUniteLegale_utf8.csv', 
    chunksize=100000,
    dtype=str)

In [17]:
start_time = time.time()

for i, df_unite_legale in enumerate(df_iterator):
    print(i)
    df_unite_legale = df_unite_legale[[
            "siren",
            "dateCreationUniteLegale",
            "sigleUniteLegale",
            "prenom1UniteLegale",
            "identifiantAssociationUniteLegale",
            "trancheEffectifsUniteLegale",
            "dateDernierTraitementUniteLegale",
            "categorieEntreprise",
            "etatAdministratifUniteLegale",
            "nomUniteLegale",
            "nomUsageUniteLegale",
            "denominationUniteLegale",
            "categorieJuridiqueUniteLegale",
            "activitePrincipaleUniteLegale",
            "economieSocialeSolidaireUniteLegale",
        ]]
    # Rename columns
    df_unite_legale = df_unite_legale.rename(
        columns={
            "dateCreationUniteLegale": "date_creation_unite_legale",
            "sigleUniteLegale": "sigle",
            "prenom1UniteLegale": "prenom",
            "trancheEffectifsUniteLegale": "tranche_effectif_salarie_unite_legale",
            "dateDernierTraitementUniteLegale": "date_mise_a_jour_unite_legale",
            "categorieEntreprise": "categorie_entreprise",
            "etatAdministratifUniteLegale":"etat_administratif_unite_legale",
            "nomUniteLegale": "nom",
            "nomUsageUniteLegale": "nom_usage",
            "denominationUniteLegale": "nom_raison_sociale",
            "categorieJuridiqueUniteLegale": "nature_juridique_unite_legale",
            "activitePrincipaleUniteLegale": "activite_principale_unite_legale",
            "economieSocialeSolidaireUniteLegale":"economie_sociale_solidaire_unite_legale",
            "identifiantAssociationUniteLegale":"identifiant_association_unite_legale",
        }
    )
    
    df_unite_legale.to_sql("unite_legale", connection, if_exists='append', index=False)
stats()
del df_unite_legale

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
--- 207.61528992652893 seconds ---
used memory : 20.3Go


### Add nom_complet

In [18]:
def get_nom_complet(nature_juridique_unite_legale, sigle, prenom, nom, nom_usage, nom_raison_sociale):
    if nature_juridique_unite_legale == "1000":
        if sigle is not None:
            if (prenom is not None) & (nom is not None):
                if(nom_usage is not None):
                    return (
                        prenom.lower()
                        + " "
                        + nom_usage.lower()
                        + " ("
                        + nom.lower()
                        + ", "
                        + sigle.lower()
                        + ")"
                    )
                else:
                    return (
                        prenom.lower()
                        + " "
                        + nom.lower()
                        + " ("
                        + sigle.lower()
                        + ")"
                    )
            else:
                return None
        else:
            if (prenom is not None) & (nom is not None):
                if(nom_usage is not None):
                    return (
                        prenom.lower()
                        + " "
                        + nom_usage.lower()
                        + " ("
                        + nom.lower()
                        + ")"
                    )
                else:
                    return prenom.lower() + " " + nom.lower()
            else:
                return None
    else:
        if(sigle is not None):
            if(nom_raison_sociale is not None):
                return nom_raison_sociale.lower() + " (" + sigle.lower() + ")"
            else:
                return None
        else:
            if(nom_raison_sociale is not None):
                return nom_raison_sociale.lower()
            else:
                return None

### Add entrepreneur individuel and section activité principale

In [19]:
def get_is_ei(nature_juridique_unite_legale):
    if(nature_juridique_unite_legale in ["1", "10", "1000"]):
        return 1
    else:
        return 0


In [20]:
sections_NAF = {
"01":"A","02":"A","03":"A","05":"B","06":"B","07":"B","08":"B","09":"B","10":"C","11":"C","12":"C","13":"C","14":"C",
 "15":"C","16":"C","17":"C","18":"C","19":"C","20":"C","21":"C","22":"C","23":"C","24":"C","25":"C","26":"C","27":"C",
 "28":"C","29":"C","30":"C","31":"C","32":"C","33":"C","35":"D","36":"E","37":"E","38":"E","39":"E","41":"F","42":"F",
 "43":"F","45":"G","46":"G","47":"G","49":"H","50":"H","51":"H","52":"H","53":"H","55":"I","56":"I","58":"J","59":"J",
 "60":"J","61":"J","62":"J","63":"J","64":"K","65":"K","66":"K","68":"L","69":"M","70":"M","71":"M","72":"M","73":"M",
 "74":"M","75":"M","77":"N","78":"N","79":"N","80":"N","81":"N","82":"N","84":"O","85":"P","86":"Q","87":"Q","88":"Q",
 "90":"R","91":"R","92":"R","93":"R","94":"S","95":"S","96":"S","97":"T","98":"T","99":"U"
}

In [21]:
def get_section(activite_principale_unite_legale):
    if(activite_principale_unite_legale is not None):
        code_naf = activite_principale_unite_legale[:2]
        section_activite_principale = sections_NAF[code_naf] if code_naf in sections_NAF else None
        return section_activite_principale
    else:
        return None

## Établissements

In [22]:
# Create list of departement zip codes
all_deps = [
    *"-0".join(list(str(x) for x in range(0, 10))).split("-")[1:],
    *list(str(x) for x in range(10, 20)),
    *["2A", "2B"],
    *list(str(x) for x in range(21, 96)),
    *"-7510".join(list(str(x) for x in range(0, 10))).split("-")[1:],
    *"-751".join(list(str(x) for x in range(10, 21))).split("-")[1:],
    *["971", "972", "973", "974", "976"],
    *[""],
]
# Remove Paris zip code
all_deps.remove("75")

In [23]:
#all_deps = ["23"]

In [24]:
cursor.execute(f'''DROP TABLE IF EXISTS siret''')
cursor.execute(f'''CREATE TABLE IF NOT EXISTS siret
        (
        id INTEGER NOT NULL PRIMARY KEY,
        siren,
        siret,
        date_creation,
        tranche_effectif_salarie,
        activite_principale_registre_metier,
        is_siege,
        numero_voie,
        type_voie,
        libelle_voie,
        code_postal,
        libelle_cedex,
        libelle_commune,
        commune,
        complement_adresse,
        complement_adresse_2,
        numero_voie_2,
        indice_repetition_2,
        type_voie_2,
        libelle_voie_2,
        commune_2,
        libelle_commune_2,
        cedex_2,
        libelle_cedex_2,
        cedex,
        date_debut_activite,
        distribution_speciale,
        distribution_speciale_2,
        etat_administratif_etablissement,
        enseigne_1,
        enseigne_2,
        enseigne_3,
        activite_principale,
        indice_repetition,
        nom_commercial,
        libelle_commune_etranger,
        code_pays_etranger,
        libelle_pays_etranger,
        libelle_commune_etranger_2,
        code_pays_etranger_2,
        libelle_pays_etranger_2,
        longitude,
        latitude,
        geo_adresse,
        geo_id)
        ''')

In [25]:
%%time
# Upload geo data by departement
for dep in all_deps:
    start_time = time.time()
    url = "https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_" + dep + ".csv.gz"
    print(url)
    df_dep = pd.read_csv(
        url,
        compression="gzip",
        dtype=str,
        usecols=[
            "siren",
            "siret",
            "dateCreationEtablissement",
            "trancheEffectifsEtablissement",
            "activitePrincipaleRegistreMetiersEtablissement",
            "etablissementSiege",
            "numeroVoieEtablissement",
            "libelleVoieEtablissement",
            "codePostalEtablissement",
            "libelleCommuneEtablissement",
            "libelleCedexEtablissement",
            "typeVoieEtablissement",
            "codeCommuneEtablissement",
            "codeCedexEtablissement",
            "complementAdresseEtablissement",
            "distributionSpecialeEtablissement",
            "complementAdresse2Etablissement",
            "indiceRepetition2Etablissement",
            "libelleCedex2Etablissement",
            "codeCedex2Etablissement",
            "numeroVoie2Etablissement",
            "typeVoie2Etablissement",
            "libelleVoie2Etablissement",
            "codeCommune2Etablissement",
            "libelleCommune2Etablissement",
            "distributionSpeciale2Etablissement",
            "dateDebut",
            "etatAdministratifEtablissement",
            "enseigne1Etablissement",
            "enseigne1Etablissement",
            "enseigne2Etablissement",
            "enseigne3Etablissement",
            "denominationUsuelleEtablissement",
            "activitePrincipaleEtablissement",
            "geo_adresse",
            "geo_id",
            "longitude",
            "latitude",
            "indiceRepetitionEtablissement",
            "libelleCommuneEtrangerEtablissement",
            "codePaysEtrangerEtablissement",
            "libellePaysEtrangerEtablissement",
            "libelleCommuneEtranger2Etablissement",
            "codePaysEtranger2Etablissement",
            "libellePaysEtranger2Etablissement",
        ],
    )
    df_dep = df_dep.rename(
        columns={
            "dateCreationEtablissement": "date_creation",
            "trancheEffectifsEtablissement": "tranche_effectif_salarie",
            "activitePrincipaleRegistreMetiersEtablissement": "activite_principale_registre_metier",
            "etablissementSiege": "is_siege",
            "numeroVoieEtablissement": "numero_voie",
            "typeVoieEtablissement": "type_voie",
            "libelleVoieEtablissement": "libelle_voie",
            "codePostalEtablissement": "code_postal",
            "libelleCedexEtablissement": "libelle_cedex",
            "libelleCommuneEtablissement": "libelle_commune",
            "codeCommuneEtablissement": "commune",
            "complementAdresseEtablissement": "complement_adresse",
            "complementAdresse2Etablissement": "complement_adresse_2",
            "numeroVoie2Etablissement": "numero_voie_2",
            "indiceRepetition2Etablissement": "indice_repetition_2",
            "typeVoie2Etablissement": "type_voie_2",
            "libelleVoie2Etablissement": "libelle_voie_2",
            "codeCommune2Etablissement": "commune_2",
            "libelleCommune2Etablissement": "libelle_commune_2",
            "codeCedex2Etablissement": "cedex_2",
            "libelleCedex2Etablissement": "libelle_cedex_2",
            "codeCedexEtablissement": "cedex",
            "dateDebut": "date_debut_activite",
            "distributionSpecialeEtablissement": "distribution_speciale",
            "distributionSpeciale2Etablissement": "distribution_speciale_2",
            "etatAdministratifEtablissement": "etat_administratif_etablissement",
            "enseigne1Etablissement": "enseigne_1",
            "enseigne2Etablissement": "enseigne_2",
            "enseigne3Etablissement": "enseigne_3",
            "activitePrincipaleEtablissement": "activite_principale",
            "indiceRepetitionEtablissement": "indice_repetition",
            "denominationUsuelleEtablissement": "nom_commercial",
            "libelleCommuneEtrangerEtablissement": "libelle_commune_etranger",
            "codePaysEtrangerEtablissement": "code_pays_etranger",
            "libellePaysEtrangerEtablissement": "libelle_pays_etranger",
            "libelleCommuneEtranger2Etablissement": "libelle_commune_etranger_2",
            "codePaysEtranger2Etablissement": "code_pays_etranger_2",
            "libellePaysEtranger2Etablissement": "libelle_pays_etranger_2",
        }
    )
    stats()
  
    start_time = time.time()
    df_dep.to_sql("siret", connection, if_exists='append', index=False)
    connection.commit()
    stats()
del df_dep

https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_01.csv.gz
--- 1.6804656982421875 seconds ---
used memory : 20.5Go
--- 4.794612407684326 seconds ---
used memory : 20.5Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_02.csv.gz
--- 1.1881725788116455 seconds ---
used memory : 20.6Go
--- 3.546485662460327 seconds ---
used memory : 20.5Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_03.csv.gz
--- 0.929689884185791 seconds ---
used memory : 20.4Go
--- 2.9710922241210938 seconds ---
used memory : 20.5Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_04.csv.gz
--- 0.765876293182373 seconds ---
used memory : 20.6Go
--- 2.066420793533325 seconds ---
used memory : 20.4Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_05.csv.gz
--- 0.6505546569824219 seconds ---
used memory : 20.4Go
--- 2.0766892433166504 seconds ---
used memory : 20.4Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_06.csv.gz
--- 5.025439977645874 seconds ---
used memo

--- 0.7796773910522461 seconds ---
used memory : 20.6Go
--- 2.003527879714966 seconds ---
used memory : 20.6Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_47.csv.gz
--- 1.2276573181152344 seconds ---
used memory : 20.7Go
--- 3.4172842502593994 seconds ---
used memory : 20.7Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_48.csv.gz
--- 0.41557979583740234 seconds ---
used memory : 20.6Go
--- 0.8943665027618408 seconds ---
used memory : 20.7Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_49.csv.gz
--- 2.1963443756103516 seconds ---
used memory : 20.7Go
--- 6.268176078796387 seconds ---
used memory : 20.7Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_50.csv.gz
--- 1.899909496307373 seconds ---
used memory : 20.7Go
--- 4.14769983291626 seconds ---
used memory : 20.7Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_51.csv.gz
--- 2.2191100120544434 seconds ---
used memory : 20.7Go
--- 5.384109258651733 seconds ---
used memory : 20.7G

--- 14.478434562683105 seconds ---
used memory : 21.1Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_94.csv.gz
--- 3.9279637336730957 seconds ---
used memory : 21.1Go
--- 12.332377672195435 seconds ---
used memory : 21.2Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_95.csv.gz
--- 2.762077808380127 seconds ---
used memory : 20.9Go
--- 9.063505172729492 seconds ---
used memory : 21.1Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_75101.csv.gz
--- 0.6094448566436768 seconds ---
used memory : 20.6Go
--- 1.641188383102417 seconds ---
used memory : 20.6Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_75102.csv.gz
--- 0.5875587463378906 seconds ---
used memory : 20.6Go
--- 2.0779967308044434 seconds ---
used memory : 20.6Go
https://files.data.gouv.fr/geo-sirene/last/dep/geo_siret_75103.csv.gz
--- 0.45946788787841797 seconds ---
used memory : 20.6Go
--- 1.3860211372375488 seconds ---
used memory : 20.6Go
https://files.data.gouv.fr/geo-sirene/last

### Add Adresse Compléte

In [26]:
def get_adresse_complete(complement_adresse, numero_voie, indice_repetition, type_voie, libelle_voie, libelle_commune, libelle_cedex, distribution_speciale, commune, cedex, libelle_commune_etranger, libelle_pays_etranger):    
    col_list = [complement_adresse, numero_voie, indice_repetition, type_voie, libelle_voie, distribution_speciale]
    adresse = ""
    for column in col_list:
        if(column is not None):
            adresse = adresse + " " + column if column is not None else ""
    if cedex is None:
        if commune is None:
            adresse =  adresse
        else:
            adresse = adresse + " " + commune + " " + libelle_commune
    else:
        adresse = adresse + " " + cedex + " " + libelle_cedex
    etranger_list = [libelle_commune_etranger, libelle_pays_etranger]
    for column in etranger_list:
        if(column is not None):
            adresse = adresse + " " + column if column is not None else ""
    return adresse.strip()

### Add dep/coord columns

In [27]:
def get_departement(commune):
    departement = str(commune)[:3] if str(commune)[:2]== "97" else (None if commune is None else str(commune)[:2])
    return departement

In [28]:
def get_coordonnees(longitude, latitude):
    coordonnees = None if (longitude is None) or (latitude is None) else f"{latitude},{longitude}"
    return coordonnees

### Add enseignes

In [29]:
def get_enseignes(enseigne_1, enseigne_2, enseigne_3, nom_commercial):
    return ' '.join(list(filter(None,set([enseigne_1, enseigne_2, enseigne_3, nom_commercial]))))

### Nombre d'établissements

In [30]:
cursor.execute(f'''DROP TABLE IF EXISTS count_etab''')

In [31]:
cursor.execute('''CREATE TABLE count_etab (siren VARCHAR(10), count INTEGER)''')

In [32]:
cursor.execute('''
                CREATE UNIQUE INDEX index_count_siren
                ON count_etab (siren);
                ''')

In [34]:
start_time = time.time()
# Add etab count
cursor.execute('''INSERT INTO count_etab (siren, count) SELECT siren, count(*) as count FROM siret GROUP BY siren;''')
stats()


--- 27.57229447364807 seconds ---
used memory : 20.5Go


In [66]:
cursor.execute(f'''DROP TABLE IF EXISTS count_etab_ouvert''')

In [67]:
cursor.execute('''CREATE TABLE count_etab_ouvert (siren VARCHAR(10), count INTEGER)''')

In [68]:
cursor.execute('''
                CREATE UNIQUE INDEX index_count_ouvert_siren
                ON count_etab_ouvert (siren);
                ''')

In [69]:
start_time = time.time()
# Add etab count
cursor.execute('''INSERT INTO count_etab_ouvert (siren, count) SELECT siren, count(*) as count FROM siret WHERE etat_administratif_etablissement = 'A' GROUP BY siren;''')
stats()


--- 15.474198818206787 seconds ---
used memory : 19.8Go


### Create table with only siege

In [39]:
cursor.execute(f'''DROP TABLE IF EXISTS siretsiege''')
cursor.execute(f'''CREATE TABLE IF NOT EXISTS siretsiege
        (
        id INTEGER NOT NULL PRIMARY KEY,
        siren,
        siret,
        date_creation,
        tranche_effectif_salarie,
        activite_principale_registre_metier,
        is_siege,
        numero_voie,
        type_voie,
        libelle_voie,
        code_postal,
        libelle_cedex,
        libelle_commune,
        commune,
        complement_adresse,
        complement_adresse_2,
        numero_voie_2,
        indice_repetition_2,
        type_voie_2,
        libelle_voie_2,
        commune_2,
        libelle_commune_2,
        cedex_2,
        libelle_cedex_2,
        cedex,
        date_debut_activite,
        distribution_speciale,
        distribution_speciale_2,
        etat_administratif_etablissement,
        enseigne_1,
        enseigne_2,
        enseigne_3,
        activite_principale,
        indice_repetition,
        nom_commercial,
        libelle_commune_etranger,
        code_pays_etranger,
        libelle_pays_etranger,
        libelle_commune_etranger_2,
        code_pays_etranger_2,
        libelle_pays_etranger_2,
        longitude,
        latitude,
        geo_adresse,
        geo_id)
''')
cursor.execute('''INSERT INTO siretsiege (
        siren,
        siret,
        date_creation,
        tranche_effectif_salarie,
        activite_principale_registre_metier,
        is_siege,
        numero_voie,
        type_voie,
        libelle_voie,
        code_postal,
        libelle_cedex,
        libelle_commune,
        commune,
        complement_adresse,
        complement_adresse_2,
        numero_voie_2,
        indice_repetition_2,
        type_voie_2,
        libelle_voie_2,
        commune_2,
        libelle_commune_2,
        cedex_2,
        libelle_cedex_2,
        cedex,
        date_debut_activite,
        distribution_speciale,
        distribution_speciale_2,
        etat_administratif_etablissement,
        enseigne_1,
        enseigne_2,
        enseigne_3,
        activite_principale,
        indice_repetition,
        nom_commercial,
        libelle_commune_etranger,
        code_pays_etranger,
        libelle_pays_etranger,
        libelle_commune_etranger_2,
        code_pays_etranger_2,
        libelle_pays_etranger_2,
        longitude,
        latitude,
        geo_adresse,
        geo_id) 
    SELECT
        siren,
        siret,
        date_creation,
        tranche_effectif_salarie,
        activite_principale_registre_metier,
        is_siege,
        numero_voie,
        type_voie,
        libelle_voie,
        code_postal,
        libelle_cedex,
        libelle_commune,
        commune,
        complement_adresse,
        complement_adresse_2,
        numero_voie_2,
        indice_repetition_2,
        type_voie_2,
        libelle_voie_2,
        commune_2,
        libelle_commune_2,
        cedex_2,
        libelle_cedex_2,
        cedex,
        date_debut_activite,
        distribution_speciale,
        distribution_speciale_2,
        etat_administratif_etablissement,
        enseigne_1,
        enseigne_2,
        enseigne_3,
        activite_principale,
        indice_repetition,
        nom_commercial,
        libelle_commune_etranger,
        code_pays_etranger,
        libelle_pays_etranger,
        libelle_commune_etranger_2,
        code_pays_etranger_2,
        libelle_pays_etranger_2,
        longitude,
        latitude,
        geo_adresse,
        geo_id
    FROM siret
    WHERE is_siege = 'true';
''')

In [40]:
cursor.execute('''
                CREATE INDEX index_siret_siren
                ON siretsiege (siren);
                ''')

In [41]:
def dict_from_row(row):
    return dict(zip(row.keys(), row))  

In [60]:
def process_res(res):
    arr = []
    for result in res:            
        mydict = {}
        mydict['siege'] = {}
        for item in result:
            if(item.startswith('ul_')):
                mydict[item[3:]] = result[item]
            elif(item.startswith('st_')):
                mydict['siege'][item[3:]] = result[item]

        mydict['nom_complet'] = get_nom_complet(
            result['ul_nature_juridique'],
            result['sigle'],
            result['prenom'],
            result['nom'],
            result['nom_usage'],
            result['ul_nom_raison_sociale']
        )    
        mydict['is_entrepreneur_individuel'] = get_is_ei(
            result['ul_nature_juridique']
        )
        mydict['section_activite_principale'] = get_section(
            result['ul_activite_principale']
        )
        mydict['siege']['departement'] = get_departement(
            result['st_commune']
        )
        arr.append(mydict)
    return arr
    

In [43]:
global_stats()

--- 1548.3741827011108 seconds ---
used memory : 20.5Go


In [44]:
chunk_size = 1500

In [45]:
for row in cursor.execute('''SELECT count(*) FROM siretsiege;'''):
    nb_iter = int(row[0]) / chunk_size + 1

In [46]:
int(nb_iter)

15456

In [47]:
start_time = time.time()

for i in range(int(nb_iter)):
    cursor.execute(f'''
        SELECT 
            ul.siren as ul_siren,
            st.siret as st_siret,
            st.date_creation as st_date_creation,
            st.tranche_effectif_salarie as st_tranche_effectif_salarie,
            st.date_debut_activite as st_date_debut_activite,
            st.etat_administratif_etablissement as st_etat_administratif,
            st.activite_principale as st_activite_principale,
            st.complement_adresse as st_complement_adresse,
            st.numero_voie as st_numero_voie,
            st.indice_repetition as st_indice_repetition,
            st.type_voie as st_type_voie,
            st.libelle_voie as st_libelle_voie,
            st.distribution_speciale as st_distribution_speciale,
            st.cedex as st_cedex,
            st.libelle_cedex as st_libelle_cedex,
            st.commune as st_commune,
            st.libelle_commune as st_libelle_commune,
            st.code_pays_etranger as st_code_pays_etranger,
            st.libelle_commune_etranger as st_libelle_commune_etranger,
            st.libelle_pays_etranger as st_libelle_pays_etranger,
            st.code_postal as st_code_postal,
            st.geo_id as st_geo_id,
            st.longitude as st_longitude,
            st.latitude as st_latitude,
            st.activite_principale_registre_metier as st_activite_principale_registre_metier,
            ul.date_creation_unite_legale as ul_date_creation,
            ul.tranche_effectif_salarie_unite_legale as ul_tranche_effectif_salarie,
            ul.date_mise_a_jour_unite_legale as ul_date_mise_a_jour,
            ul.categorie_entreprise as ul_categorie_entreprise,
            ul.etat_administratif_unite_legale as ul_etat_administratif,
            ul.nom_raison_sociale as ul_nom_raison_sociale,
            ul.nature_juridique_unite_legale as ul_nature_juridique,
            ul.activite_principale_unite_legale as ul_activite_principale,
            ul.economie_sociale_solidaire_unite_legale as ul_economie_sociale_solidaire,
            (SELECT count FROM count_etab ce WHERE ce.siren = st.siren) as ul_nombre_etablissements,
            (SELECT count FROM count_etab_ouvert ceo WHERE ceo.siren = st.siren) as ul_nombre_etablissements_ouverts,
            ul.sigle,
            ul.prenom,
            ul.nom,
            ul.nom_usage,
            st.is_siege
        FROM
            siretsiege st
        LEFT JOIN unite_legale ul 
        ON
            ul.siren = st.siren
        WHERE id > {i*1500}
        LIMIT 1500
    ''')

    res = cursor.fetchall()
    columns = tuple([x[0] for x in cursor.description])
    res = tuple(
        [{column: val for column, val in zip(columns, x)} for x in res]
    )
    res2 = process_res(res)
    if(i%100 == 0):
        print(i)

stats()

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
--- 475.77531480789185 seconds ---
used memory : 20.5Go


In [48]:
global_stats()

--- 2024.2443099021912 seconds ---
used memory : 20.5Go
